# Study the Data and Create Data Catalogs

## 1. Start Spark and locate the raw files

In [1]:
import os
import sys
from functools import reduce
from pathlib import Path

from data_intensive_computing.spark_env import configure_windows_spark_env

# Must run before Spark starts; configures Java, Hadoop, Python worker, and loopback on Windows.
configure_windows_spark_env()

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import ArrayType, StringType, StructField, StructType

print("Python:", sys.executable)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))

Python: D:\Master\data intensive\data-intensive-computing\.venv\Scripts\python.exe
JAVA_HOME: D:\Eclipse Adoptium\jdk-17.0.20.101-hotspot
HADOOP_HOME: D:\Master\data intensive\data-intensive-computing\tools\hadoop
PYSPARK_PYTHON: C:\Users\12547\AppData\Roaming\uv\python\cpython-3.14-windows-x86_64-none\python.exe


In [2]:
# Set up Spark session 
builder = (
    SparkSession.builder
    .appName("Lab1-Task1-Data-Catalog")
    .master("local[4]")
    .config("spark.driver.memory", "3g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.local.dir", os.environ["SPARK_LOCAL_DIRS"])
    .config("spark.driver.extraJavaOptions", f'-Djava.io.tmpdir={os.environ["SPARK_JAVA_TMP_DIR"]}')
    .config("spark.jars.ivy", os.environ["SPARK_IVY_DIR"])
    .config("spark.sql.warehouse.dir", os.environ["SPARK_WAREHOUSE_DIR"])
    .config("spark.hadoop.hadoop.tmp.dir", os.environ["HADOOP_TMP_DIR"])
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.pyspark.python", os.environ["PYSPARK_PYTHON"])
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "America/New_York")
spark.sparkContext.setLogLevel("WARN")

def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not find data/raw")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data" / "raw"
print("Spark version:", spark.version)
print("Raw data directory:", DATA_DIR)

D:\Master\data intensive\data-intensive-computing\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0
Raw data directory: D:\Master\data intensive\data-intensive-computing\data\raw


## 2. Load the four raw datasets

In [3]:
from urllib.parse import quote
print("starting taxi read...")
files = sorted(str(f) for f in DATA_DIR.glob("yellow_tripdata_2024-*.parquet"))
print(files)
taxi = spark.read.parquet(*files)
print(taxi.count())
print("taxi read done")
weather = (
    spark.read.option("header", True).option("inferSchema", True)
    .csv(str(DATA_DIR / "weather.csv"))
)
air_quality = (
    spark.read.option("header", True).option("inferSchema", False)
    .csv(str(DATA_DIR / "hourly_88101_2024.csv"))
)
taxi_zones = (
    spark.read.option("header", True).option("inferSchema", True)
    .csv(str(DATA_DIR / "taxi_zone_lookup.csv"))
)
datasets = {
    "Taxi Trips": taxi,
    "Weather": weather,
    "Air Quality": air_quality,
    "Taxi Zone Lookup": taxi_zones,
}

starting taxi read...
['D:\\Master\\data intensive\\data-intensive-computing\\data\\raw\\yellow_tripdata_2024-01.parquet', 'D:\\Master\\data intensive\\data-intensive-computing\\data\\raw\\yellow_tripdata_2024-02.parquet', 'D:\\Master\\data intensive\\data-intensive-computing\\data\\raw\\yellow_tripdata_2024-03.parquet']
9554778
taxi read done


## 3. Inspect schema, size, and sample rows

In [4]:
dataset_counts = {}
for dataset_name, df in datasets.items():
    print("=" * 80)
    print(dataset_name)
    dataset_counts[dataset_name] = df.count()
    print("Rows:", dataset_counts[dataset_name])
    print("Columns:", len(df.columns))
    df.printSchema()
    df.show(5, truncate=False)

Taxi Trips
Rows: 9554778
Columns: 19
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+-----

## 4. Check candidate primary keys


In [5]:
# Check candidate keys for each dataset
def check_candidate_key(df, dataset_name, key_columns):
    missing_conditions = [
        F.col(column).isNull() | (F.trim(F.col(column).cast("string")) == "")
        for column in key_columns
    ]
    missing_key_rows = df.filter(reduce(lambda left, right: left | right, missing_conditions)).count()
    duplicate_groups = (
        df.groupBy(*key_columns).count().filter(F.col("count") > 1)
    )
    result = {
        "dataset": dataset_name,
        "candidate_key": " + ".join(key_columns),
        "missing_key_rows": missing_key_rows,
        "duplicate_key_groups": duplicate_groups.count(),
    }
    print(result)
    duplicate_groups.orderBy(F.desc("count")).show(10, truncate=False)
    return result

weather_for_key = weather.withColumn(
    "_catalog_observation_timestamp",
    F.make_timestamp("year", "month", "day", "hour", F.lit(0), F.lit(0)),
)
air_for_key = air_quality.withColumn(
    "_catalog_observation_timestamp",
    F.to_timestamp(F.concat_ws(" ", F.col("Date Local"), F.col("Time Local")), "yyyy-MM-dd HH:mm"),
)

In [6]:
key_check_results = [
    check_candidate_key(
        taxi,
        "Taxi Trips",
        [
            "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
            "PULocationID", "DOLocationID", "trip_distance", "total_amount",
        ],
    ),
    check_candidate_key(weather_for_key, "Weather", ["_catalog_observation_timestamp"]),
    check_candidate_key(
        air_for_key,
        "Air Quality",
        [
            "State Code", "County Code", "Site Num", "Parameter Code",
            "POC", "_catalog_observation_timestamp", "Method Code",
        ],
    ),
    check_candidate_key(taxi_zones, "Taxi Zone Lookup", ["LocationID"]),
]

{'dataset': 'Taxi Trips', 'candidate_key': 'VendorID + tpep_pickup_datetime + tpep_dropoff_datetime + PULocationID + DOLocationID + trip_distance + total_amount', 'missing_key_rows': 0, 'duplicate_key_groups': 1}
+--------+--------------------+---------------------+------------+------------+-------------+------------+-----+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|PULocationID|DOLocationID|trip_distance|total_amount|count|
+--------+--------------------+---------------------+------------+------------+-------------+------------+-----+
|2       |2024-02-08 18:23:50 |2024-02-08 18:38:14  |162         |230         |0.81         |23.16       |2    |
+--------+--------------------+---------------------+------------+------------+-------------+------------+-----+

{'dataset': 'Weather', 'candidate_key': '_catalog_observation_timestamp', 'missing_key_rows': 0, 'duplicate_key_groups': 1}
+------------------------------+-----+
|_catalog_observation_timestamp|count|
+------------------

## 5. Create the Data Catalog

In [7]:
catalog_schema = StructType([
    StructField("dataset", StringType(), False),
    StructField("primary_entity", StringType(), False),
    StructField("primary_key", StringType(), False),
    StructField("join_attributes", ArrayType(StringType()), False),
    StructField("temporal_attributes", ArrayType(StringType()), False),
    StructField("categorical_attributes", ArrayType(StringType()), False),
    StructField("growth_over_time", StringType(), False),
])

catalog_rows = [
    (
        "Taxi Trips",
        "One completed yellow-taxi trip",
        "No native primary key. Candidate composite: VendorID + tpep_pickup_datetime + tpep_dropoff_datetime + PULocationID + DOLocationID + trip_distance + total_amount",
        ["PULocationID", "DOLocationID", "tpep_pickup_datetime"],
        ["tpep_pickup_datetime", "tpep_dropoff_datetime"],
        ["VendorID", "RatecodeID", "store_and_fwd_flag", "PULocationID", "DOLocationID", "payment_type"],
        "New trip records are appended as pickup and dropoff time advance",
    ),
    (
        "Weather",
        "Weather conditions observed during one hour",
        "year + month + day + hour",
        ["year", "month", "day", "hour"],
        ["year", "month", "day", "hour"],
        [
            "temp_source", "rhum_source", "prcp_source", "snwd_source",
            "wdir_source", "wspd_source", "wpgt_source", "pres_source",
            "cldc", "cldc_source", "coco", "coco_source",
        ],
        "New hourly observations are appended as year, month, day, and hour advance",
    ),
    (
        "Air Quality",
        "One PM2.5 measurement from a monitoring instrument at a site and hour",
        "State Code + County Code + Site Num + Parameter Code + POC + Date Local + Time Local + Method Code",
        ["Date Local", "Time Local", "County Name", "Latitude", "Longitude"],
        ["Date Local", "Time Local", "Date GMT", "Time GMT", "Date of Last Change"],
        [
            "State Code", "County Code", "Site Num", "Parameter Code", "POC",
            "Datum", "Parameter Name", "Units of Measure", "Qualifier",
            "Method Type", "Method Code", "Method Name", "State Name", "County Name",
        ],
        "New measurements are appended as local/GMT time advances; Date of Last Change advances when records are corrected",
    ),
    (
        "Taxi Zone Lookup",
        "One TLC taxi zone",
        "LocationID",
        ["LocationID", "Borough"],
        [],
        ["LocationID", "Borough", "Zone", "service_zone"],
        "Not expected to grow regularly; zones may occasionally be added or revised",
    ),
]

data_catalog = spark.createDataFrame(catalog_rows, schema=catalog_schema)
data_catalog.show(truncate=False, vertical=True)

-RECORD 0----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 dataset                | Taxi Trips                                                                                                                                                                   
 primary_entity         | One completed yellow-taxi trip                                                                                                                                               
 primary_key            | No native primary key. Candidate composite: VendorID + tpep_pickup_datetime + tpep_dropoff_datetime + PULocationID + DOLocationID + trip_distance + total_amount             
 join_attributes        | [PULocationID, DOLocationID, tpep_pickup_datetime]                                                                                                                           


In [8]:
import subprocess
import textwrap
try:
    from PIL import Image, ImageDraw, ImageFont
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "Pillow"])
    from PIL import Image, ImageDraw, ImageFont
from IPython.display import Image as NotebookImage, display as notebook_display

catalog_headers = [
    "Dataset", "Primary entity", "Primary key", "Join attributes",
    "Temporal attributes", "Categorical attributes", "Growth over time",
]
catalog_image_rows = []
for catalog_record in data_catalog.collect():
    catalog_image_rows.append([
        catalog_record.dataset,
        catalog_record.primary_entity,
        catalog_record.primary_key,
        ", ".join(catalog_record.join_attributes) if catalog_record.join_attributes else "None",
        ", ".join(catalog_record.temporal_attributes) if catalog_record.temporal_attributes else "None",
        ", ".join(catalog_record.categorical_attributes) if catalog_record.categorical_attributes else "None",
        catalog_record.growth_over_time,
    ])

column_widths = [260, 410, 720, 520, 520, 790, 610]
margin = 70
title_height = 105
header_height = 86
cell_padding_x = 18
cell_padding_y = 18
line_height = 32
regular_font_path = "/System/Library/Fonts/Supplemental/Arial.ttf"
bold_font_path = "/System/Library/Fonts/Supplemental/Arial Bold.ttf"
title_font = ImageFont.truetype(bold_font_path, 46)
header_font = ImageFont.truetype(bold_font_path, 25)
body_font = ImageFont.truetype(regular_font_path, 24)
dataset_font = ImageFont.truetype(bold_font_path, 25)

def wrap_catalog_cell(value, width, font):
    average_character_width = max(
        font.getlength("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz") / 52, 1
    )
    maximum_characters = max(8, int((width - 2 * cell_padding_x) / average_character_width))
    wrapped_lines = []
    for paragraph in str(value).splitlines() or [""]:
        wrapped_lines.extend(
            textwrap.wrap(
                paragraph, width=maximum_characters,
                break_long_words=False, break_on_hyphens=False,
            ) or [""]
        )
    return wrapped_lines

wrapped_catalog_rows = []
catalog_row_heights = []
for catalog_row in catalog_image_rows:
    wrapped_row = [
        wrap_catalog_cell(value, width, dataset_font if column_index == 0 else body_font)
        for column_index, (value, width) in enumerate(zip(catalog_row, column_widths))
    ]
    wrapped_catalog_rows.append(wrapped_row)
    catalog_row_heights.append(
        max(len(lines) for lines in wrapped_row) * line_height + 2 * cell_padding_y
    )

canvas_width = 2 * margin + sum(column_widths)
canvas_height = 2 * margin + title_height + header_height + sum(catalog_row_heights)
catalog_image = Image.new("RGB", (canvas_width, canvas_height), "#F7F9FC")
catalog_draw = ImageDraw.Draw(catalog_image)
catalog_draw.text((margin, margin), "Week 1 - Task 1 Data Catalog", font=title_font, fill="#172033")

current_y = margin + title_height
current_x = margin
for header, width in zip(catalog_headers, column_widths):
    catalog_draw.rectangle(
        (current_x, current_y, current_x + width, current_y + header_height),
        fill="#173F5F", outline="#D6DEE8", width=2,
    )
    header_lines = wrap_catalog_cell(header, width, header_font)
    header_y = current_y + (header_height - len(header_lines) * line_height) / 2
    for header_line in header_lines:
        catalog_draw.text((current_x + cell_padding_x, header_y), header_line, font=header_font, fill="white")
        header_y += line_height
    current_x += width

current_y += header_height
for row_index, (wrapped_row, row_height) in enumerate(zip(wrapped_catalog_rows, catalog_row_heights)):
    row_fill = "#FFFFFF" if row_index % 2 == 0 else "#EDF3F8"
    current_x = margin
    for column_index, (cell_lines, width) in enumerate(zip(wrapped_row, column_widths)):
        catalog_draw.rectangle(
            (current_x, current_y, current_x + width, current_y + row_height),
            fill=row_fill, outline="#CCD6E1", width=2,
        )
        cell_font = dataset_font if column_index == 0 else body_font
        cell_y = current_y + cell_padding_y
        for cell_line in cell_lines:
            catalog_draw.text((current_x + cell_padding_x, cell_y), cell_line, font=cell_font, fill="#172033")
            cell_y += line_height
        current_x += width
    current_y += row_height

report_directory = PROJECT_ROOT / "reports"
report_directory.mkdir(parents=True, exist_ok=True)
catalog_image_path = report_directory / "data_catalog.png"
catalog_image.save(catalog_image_path, format="PNG", optimize=True)
print(f"Saved catalog image: {catalog_image_path}")
notebook_display(NotebookImage(filename=str(catalog_image_path)))

OSError: cannot open resource